In [2]:
   import os, sys
   if os.path.basename(os.getcwd()) == "notebooks":
       os.chdir("..")
   sys.path.append(os.getcwd())
   from common import *
   print("Working folder:", os.getcwd())
   print("Features:", FEATURES)

Working folder: c:\Users\Vallabh\Documents\GitHub\ai-grid-fault-classifier
Features: ['Ia', 'Ib', 'Ic', 'Va', 'Vb', 'Vc', 'Im', 'Vm', 'Pai', 'Pav', 'Fi', 'Fv']


In [3]:
# Load the cleaned data (made by Member 1)
df = load_clean()
print("Total rows:", df.shape)

# Time-blocked split: first 80% of each class for training, last 20% for testing
train, test = blocked_split(df)

X_train, y_train = train[FEATURES], train[TARGET]
X_test,  y_test  = test[FEATURES],  test[TARGET]

print("Train rows:", X_train.shape[0])
print("Test rows: ", X_test.shape[0])
print("time_stamp used as feature?", "time_stamp" in FEATURES)
print()
print("Rows per class in training:")
print(y_train.value_counts())
print("Blocks in training data:", sorted(train["block"].unique()))

Total rows: (134019, 14)
Train rows: 107218
Test rows:  26801
time_stamp used as feature? False

Rows per class in training:
fault_types
LG      17870
LLG     17870
LLL     17870
LLLG    17870
NF      17870
LL      17868
Name: count, dtype: int64
Blocks in training data: [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]


In [4]:
import time, joblib
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

os.makedirs("models", exist_ok=True)

models = {
    "DecisionTree": DecisionTreeClassifier(max_depth=8, random_state=RANDOM_STATE),
    "RandomForest": RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE),
}

for name, m in models.items():
    start = time.time()
    m.fit(X_train, y_train)                      # learn from training data only
    path = f"models/{name}.joblib"
    joblib.dump(m, path, compress=3)             # save the trained model
    size_mb = os.path.getsize(path) / 1e6
    print(f"{name}: trained in {time.time()-start:.1f}s, "
          f"file size {size_mb:.1f} MB, training accuracy {m.score(X_train, y_train):.4f}")

DecisionTree: trained in 0.7s, file size 0.0 MB, training accuracy 1.0000
RandomForest: trained in 2.9s, file size 0.3 MB, training accuracy 1.0000


In [5]:
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

svm = make_pipeline(StandardScaler(), SVC(kernel="rbf", C=10, gamma="scale"))

start = time.time()
svm.fit(X_train.iloc[::4], y_train.iloc[::4])    # every 4th row only
path = "models/SVM_RBF.joblib"
joblib.dump(svm, path, compress=3)
models["SVM_RBF"] = svm                           # add to our dictionary

print(f"SVM_RBF: trained on {len(X_train.iloc[::4])} rows in {time.time()-start:.1f}s, "
      f"file size {os.path.getsize(path)/1e6:.1f} MB")
print("Training accuracy (on the rows it saw):", round(svm.score(X_train.iloc[::4], y_train.iloc[::4]), 4))
print("Models ready:", list(models.keys()))

SVM_RBF: trained on 26805 rows in 1.5s, file size 0.1 MB
Training accuracy (on the rows it saw): 1.0
Models ready: ['DecisionTree', 'RandomForest', 'SVM_RBF']


In [6]:
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.base import clone
import numpy as np

groups = train["block"]
cv = GroupKFold(n_splits=4)
cv_results = {}

for name, m in models.items():
    if name == "SVM_RBF":                      # SVM uses every 4th row, as before
        Xs, ys, gs = X_train.iloc[::4], y_train.iloc[::4], groups.iloc[::4]
    else:
        Xs, ys, gs = X_train, y_train, groups
    scores = cross_val_score(clone(m), Xs, ys, groups=gs, cv=cv, scoring="accuracy")
    cv_results[name] = scores
    print(f"{name:13s} per-fold accuracy: {np.round(scores, 4)}  mean = {scores.mean():.4f}")

DecisionTree  per-fold accuracy: [1.     0.9912 1.     1.    ]  mean = 0.9978
RandomForest  per-fold accuracy: [1.     0.9951 1.     1.    ]  mean = 0.9988
SVM_RBF       per-fold accuracy: [1.     0.7742 1.     1.    ]  mean = 0.9436


In [7]:
from sklearn.metrics import classification_report

Xs, ys, gs = X_train.iloc[::4], y_train.iloc[::4], groups.iloc[::4]

for fold, (tr_idx, va_idx) in enumerate(cv.split(Xs, ys, gs), start=1):
    if fold != 2:                       # look only at the weak fold
        continue
    m = clone(models["SVM_RBF"]).fit(Xs.iloc[tr_idx], ys.iloc[tr_idx])
    pred = m.predict(Xs.iloc[va_idx])
    print("Held-out block(s):", sorted(gs.iloc[va_idx].unique()))
    print(classification_report(ys.iloc[va_idx], pred, digits=3, zero_division=0))

    # compare value ranges: held-out block vs training blocks
    print("Held-out block feature ranges vs training ranges (min / max):")
    for f in ["Im", "Vm", "Pai", "Pav"]:
        print(f"{f:4s} held-out: {Xs.iloc[va_idx][f].min():10.2f} / {Xs.iloc[va_idx][f].max():10.2f}"
              f"   train: {Xs.iloc[tr_idx][f].min():10.2f} / {Xs.iloc[tr_idx][f].max():10.2f}")

Held-out block(s): [np.int64(0)]
              precision    recall  f1-score   support

          LG      0.999     0.782     0.878      1117
          LL      0.427     1.000     0.598      1117
         LLG      0.999     0.756     0.861      1117
         LLL      0.987     0.680     0.806      1117
        LLLG      1.000     0.679     0.809      1117
          NF      1.000     0.747     0.855      1117

    accuracy                          0.774      6702
   macro avg      0.902     0.774     0.801      6702
weighted avg      0.902     0.774     0.801      6702

Held-out block feature ranges vs training ranges (min / max):
Im   held-out:    9219.84 /   31855.11   train:    9348.96 /   30275.58
Vm   held-out:      18.88 /     299.17   train:      25.72 /     299.16
Pai  held-out:     -97.50 /      21.20   train:     -59.75 /      12.12
Pav  held-out:     -97.56 /      21.60   train:     -59.75 /      12.35


In [8]:
b0 = train[train["block"] == 0]
rest = train[train["block"] != 0]

lo, hi = rest["Pai"].min(), rest["Pai"].max()
outside = ((b0["Pai"] < lo) | (b0["Pai"] > hi)).mean()
print("Share of block-0 rows with Pai outside the range of blocks 1-3:", round(outside, 3))
print()
for c in CLASSES:
    g = b0[b0[TARGET] == c]
    parts = np.array_split(g["Pai"].values, 4)       # 4 time chunks inside block 0
    meds = [round(float(np.median(p)), 1) for p in parts]
    print(f"{c:5s} median Pai over time in block 0: {meds} | blocks 1-3: {rest[rest[TARGET]==c]['Pai'].median():.1f}")

Share of block-0 rows with Pai outside the range of blocks 1-3: 0.146

NF    median Pai over time in block 0: [12.1, 12.1, 12.1, 12.1] | blocks 1-3: 12.1
LG    median Pai over time in block 0: [-4.6, -4.6, -4.6, -4.6] | blocks 1-3: -4.6
LL    median Pai over time in block 0: [-35.3, -35.2, -35.2, -35.2] | blocks 1-3: -35.2
LLG   median Pai over time in block 0: [-33.4, -33.4, -33.4, -33.4] | blocks 1-3: -33.4
LLL   median Pai over time in block 0: [-59.9, -59.8, -59.8, -59.8] | blocks 1-3: -59.8
LLLG  median Pai over time in block 0: [-48.4, -48.4, -48.4, -48.4] | blocks 1-3: -48.4


In [9]:
b0 = train[train["block"] == 0]
rest = train[train["block"] != 0]

# Row is "extreme" if any of these features is outside the range seen in blocks 1-3
cols = ["Im", "Vm", "Pai", "Pav"]
extreme = np.zeros(len(b0), dtype=bool)
for f in cols:
    extreme |= ((b0[f] < rest[f].min()) | (b0[f] > rest[f].max())).values

print("Extreme rows in block 0:", extreme.sum(), "of", len(b0), f"({extreme.mean():.1%})")
print("Extreme rows per class:")
print(b0.loc[extreme, TARGET].value_counts().to_string())
print()

svm0 = clone(models["SVM_RBF"]).fit(rest[FEATURES].iloc[::4], rest[TARGET].iloc[::4])
rf0  = clone(models["RandomForest"]).fit(rest[FEATURES], rest[TARGET])

for name, m in [("SVM_RBF", svm0), ("RandomForest", rf0)]:
    pred = m.predict(b0[FEATURES])
    ok = (pred == b0[TARGET].values)
    print(f"{name:13s} accuracy on normal rows: {ok[~extreme].mean():.4f} | "
          f"on extreme rows: {ok[extreme].mean():.4f}")

Extreme rows in block 0: 5045 of 26807 (18.8%)
Extreme rows per class:
fault_types
NF      2922
LLL     1925
LLLG     142
LL        39
LLG       17

SVM_RBF       accuracy on normal rows: 0.8460 | on extreme rows: 0.4646
RandomForest  accuracy on normal rows: 0.9975 | on extreme rows: 0.9849


In [10]:
from sklearn.model_selection import GridSearchCV
import pandas as pd

Xs, ys, gs = X_train.iloc[::4], y_train.iloc[::4], groups.iloc[::4]

searches = {
    "DecisionTree": (DecisionTreeClassifier(random_state=RANDOM_STATE),
                     {"max_depth": [4, 8, 12, None]}, X_train, y_train, groups),
    "RandomForest": (RandomForestClassifier(n_jobs=-1, random_state=RANDOM_STATE),
                     {"n_estimators": [100, 200], "max_depth": [None, 10, 20]},
                     X_train, y_train, groups),
    "SVM_RBF":      (make_pipeline(StandardScaler(), SVC(kernel="rbf")),
                     {"svc__C": [1, 10, 100], "svc__gamma": ["scale", 0.01, 0.001]},
                     Xs, ys, gs),
}

best_models, summary = {}, []
for name, (est, grid, X, y, g) in searches.items():
    gsrch = GridSearchCV(est, grid, cv=GroupKFold(4), scoring="accuracy", n_jobs=-1)
    gsrch.fit(X, y, groups=g)
    r = pd.DataFrame(gsrch.cv_results_)
    fold_cols = [c for c in r.columns if c.startswith("split") and c.endswith("_test_score")]
    r["worst_fold"] = r[fold_cols].min(axis=1)
    print(f"\n=== {name} ===")
    print(r[["params", "mean_test_score", "worst_fold"]]
          .sort_values("mean_test_score", ascending=False).round(4).to_string(index=False))
    print("Best by mean:", gsrch.best_params_)
    best_models[name] = gsrch.best_estimator_
    summary.append([name, str(gsrch.best_params_), round(gsrch.best_score_, 4)])

print("\nSummary table for the report:")
print(pd.DataFrame(summary, columns=["model", "best_params", "cv_mean_accuracy"]).to_string(index=False))


=== DecisionTree ===
             params  mean_test_score  worst_fold
   {'max_depth': 8}           0.9978      0.9912
  {'max_depth': 12}           0.9978      0.9912
{'max_depth': None}           0.9978      0.9912
   {'max_depth': 4}           0.9569      0.8275
Best by mean: {'max_depth': 8}

=== RandomForest ===
                                  params  mean_test_score  worst_fold
{'max_depth': None, 'n_estimators': 100}           0.9989      0.9957
  {'max_depth': 10, 'n_estimators': 100}           0.9989      0.9957
  {'max_depth': 20, 'n_estimators': 100}           0.9989      0.9957
{'max_depth': None, 'n_estimators': 200}           0.9988      0.9951
  {'max_depth': 10, 'n_estimators': 200}           0.9988      0.9951
  {'max_depth': 20, 'n_estimators': 200}           0.9988      0.9951
Best by mean: {'max_depth': None, 'n_estimators': 100}

=== SVM_RBF ===
                                params  mean_test_score  worst_fold
  {'svc__C': 100, 'svc__gamma': 0.001}           0

In [11]:
svm_grid2 = {"svc__C": [100, 1000], "svc__gamma": [0.001, 0.0003, 0.0001]}
gs2 = GridSearchCV(make_pipeline(StandardScaler(), SVC(kernel="rbf")),
                   svm_grid2, cv=GroupKFold(4), scoring="accuracy", n_jobs=-1)
gs2.fit(Xs, ys, groups=gs)

r2 = pd.DataFrame(gs2.cv_results_)
fold_cols = [c for c in r2.columns if c.startswith("split") and c.endswith("_test_score")]
r2["worst_fold"] = r2[fold_cols].min(axis=1)
print(r2[["params", "mean_test_score", "worst_fold"]]
      .sort_values("mean_test_score", ascending=False).round(4).to_string(index=False))
print("Best by mean:", gs2.best_params_, "| earlier best mean was 0.9559")

                                params  mean_test_score  worst_fold
 {'svc__C': 100, 'svc__gamma': 0.0001}           0.9619      0.8475
{'svc__C': 1000, 'svc__gamma': 0.0001}           0.9606      0.8424
{'svc__C': 1000, 'svc__gamma': 0.0003}           0.9592      0.8369
 {'svc__C': 100, 'svc__gamma': 0.0003}           0.9592      0.8368
  {'svc__C': 100, 'svc__gamma': 0.001}           0.9559      0.8238
 {'svc__C': 1000, 'svc__gamma': 0.001}           0.9558      0.8232
Best by mean: {'svc__C': 100, 'svc__gamma': 0.0001} | earlier best mean was 0.9559


In [12]:
# Final models: DT and RF from the grid search (already fitted on all training rows),
# SVM rebuilt with the best settings from the extended search.
final_models = {
    "DecisionTree": best_models["DecisionTree"],
    "RandomForest": best_models["RandomForest"],
    "SVM_RBF": make_pipeline(StandardScaler(),
                             SVC(kernel="rbf", C=100, gamma=0.0001)).fit(X_train.iloc[::4], y_train.iloc[::4]),
}

for name, m in final_models.items():
    path = f"models/{name}.joblib"
    joblib.dump(m, path, compress=3)
    print(f"{name}: saved, {os.path.getsize(path)/1e6:.2f} MB")

# Predictions on the held-out test set, for Member 4
preds = pd.DataFrame({"y_true": y_test.values}, index=y_test.index)
for name, m in final_models.items():
    preds[name] = m.predict(X_test)
preds.to_csv("models/predictions_test.csv", index_label="row_index")

# Tuning table for the report
tuning = pd.DataFrame([
    ["DecisionTree", "max_depth=8",                 0.9978, 0.9912],
    ["RandomForest", "n_estimators=100, max_depth=None", 0.9989, 0.9957],
    ["SVM_RBF",      "C=100, gamma=0.0001",         0.9619, 0.8475],
], columns=["model", "best_params", "cv_mean_accuracy", "cv_worst_fold"])
tuning.to_csv("models/tuning_results.csv", index=False)

print("\nTest accuracy (looked at once, do not tune on this):")
for name in final_models:
    print(f"  {name:13s} {(preds[name] == preds['y_true']).mean():.4f}")

DecisionTree: saved, 0.00 MB
RandomForest: saved, 0.14 MB
SVM_RBF: saved, 0.18 MB

Test accuracy (looked at once, do not tune on this):
  DecisionTree  1.0000
  RandomForest  1.0000
  SVM_RBF       1.0000
